In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import PoissonRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from scipy.stats import poisson

df = pd.read_csv('final.csv')

In [2]:
# ── 타겟 생성 ──────────────────────────────────────────────
def get_result(row):
    if row['home_score'] > row['away_score']: return 2   # 홈 승
    elif row['home_score'] == row['away_score']: return 1 # 무승부
    else: return 0                                         # 홈 패

df['result'] = df.apply(get_result, axis=1)

In [3]:
# ── 피처 선택 (Feature Importance 상위 피처 위주) ───────────
feature_cols = [
    'points_avg_lastn_diff', 'rank_diff',
    'goals_suf_avg_lastn_diff', 'goals_avg_diff',
    'goals_suf_avg_diff', 'rank_avg_diff',
    'points_avg_lastn_home', 'rank_avg_away',
    'points_avg_lastn_away', 'rank_away'
]

X = df[feature_cols].fillna(0)
y_result = df['result']
y_home_goals = df['home_score']
y_away_goals = df['away_score']

X_train, X_test, y_train, y_test = train_test_split(X, y_result, test_size=0.2, random_state=42)
_, _, y_home_train, y_home_test = train_test_split(X, y_home_goals, test_size=0.2, random_state=42)
_, _, y_away_train, y_away_test = train_test_split(X, y_away_goals, test_size=0.2, random_state=42)

In [4]:
# ══════════════════════════════════════════════════
# 모델 1 — Random Forest Classifier
# ══════════════════════════════════════════════════
rf = RandomForestClassifier(n_estimators=200, max_depth=8,
                             random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_acc = accuracy_score(y_test, rf_pred)

print("=" * 50)
print("📊 Random Forest 결과")
print(f"Accuracy: {rf_acc:.4f}")
print(classification_report(y_test, rf_pred,
      target_names=['홈 패(0)', '무승부(1)', '홈 승(2)']))

📊 Random Forest 결과
Accuracy: 0.5930
              precision    recall  f1-score   support

      홈 패(0)       0.51      0.61      0.56       179
      무승부(1)       0.46      0.04      0.07       152
      홈 승(2)       0.64      0.84      0.72       325

    accuracy                           0.59       656
   macro avg       0.54      0.50      0.45       656
weighted avg       0.56      0.59      0.53       656



In [5]:
# ══════════════════════════════════════════════════
# 모델 2 — 포아송 회귀 (득점 수 예측 → 승리 확률 계산)
# ══════════════════════════════════════════════════
# 홈팀/어웨이팀 득점 수를 각각 포아송 회귀로 예측
poisson_home = PoissonRegressor(alpha=0.1, max_iter=300)
poisson_away = PoissonRegressor(alpha=0.1, max_iter=300)

poisson_home.fit(X_train, y_home_train)
poisson_away.fit(X_train, y_away_train)

# 예측 득점 λ (포아송 분포의 평균)
lambda_home = poisson_home.predict(X_test)
lambda_away = poisson_away.predict(X_test)

# 포아송 분포로 승/무/패 확률 계산 (최대 10골까지 고려)
def poisson_result_probs(lh, la, max_goals=10):
    probs = np.array([
        [poisson.pmf(i, lh) * poisson.pmf(j, la)
         for j in range(max_goals + 1)]
        for i in range(max_goals + 1)
    ])
    home_win = np.sum(np.tril(probs, -1))   # 홈 득점 > 어웨이 득점
    draw     = np.sum(np.diag(probs))        # 동점
    away_win = np.sum(np.triu(probs, 1))     # 어웨이 득점 > 홈 득점
    return home_win, draw, away_win

poisson_preds = []
for lh, la in zip(lambda_home, lambda_away):
    hw, d, aw = poisson_result_probs(lh, la)
    poisson_preds.append(np.argmax([aw, d, hw]))  # 0=홈패, 1=무, 2=홈승

poisson_acc = accuracy_score(y_test, poisson_preds)

print("=" * 50)
print("📊 포아송 회귀 결과")
print(f"Accuracy: {poisson_acc:.4f}")
print(classification_report(y_test, poisson_preds,
      target_names=['홈 패(0)', '무승부(1)', '홈 승(2)']))

c:\Users\jonghun\.conda\envs\facamp\Lib\site-packages\sklearn\linear_model\_linear_loss.py:356: RuntimeWarning: invalid value encountered in matmul
  X_grad = X.T @ grad_pointwise


📊 포아송 회귀 결과
Accuracy: 0.4954
              precision    recall  f1-score   support

      홈 패(0)       0.00      0.00      0.00       179
      무승부(1)       0.00      0.00      0.00       152
      홈 승(2)       0.50      1.00      0.66       325

    accuracy                           0.50       656
   macro avg       0.17      0.33      0.22       656
weighted avg       0.25      0.50      0.33       656



c:\Users\jonghun\.conda\envs\facamp\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\jonghun\.conda\envs\facamp\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\jonghun\.conda\envs\facamp\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape

In [8]:
# ══════════════════════════════════════════════════
# 모델 비교 요약
# ══════════════════════════════════════════════════
print("=" * 50)
print("모델별 Accuracy 비교")
print(f"  Random Forest : {rf_acc:.4f}")
print(f"  포아송 회귀   : {poisson_acc:.4f}")

모델별 Accuracy 비교
  Random Forest : 0.5930
  포아송 회귀   : 0.4954


In [ ]:
# randomforestclassifier는 이름에도 나와있지만
# 홈승(2) / 무승부(1) / 홈패(0) 를 예측하는 카테고리를 맞추는 것이므로
# 분류(classifier) 모델 입니다

# 원리는 이름 처럼 나무 여러개를 만들어 다수결로 결정하는 방식입니다.
# 1단계 : 배깅 (Bagging): 데이터를 여러 개로 복제
# 전체 데이터에서 중복을 허용해 랜덤 샘플링해서 나무 수만큼 데이터셋을 만듭니다.

# 전체 데이터 (1000경기)
#     ↓ 랜덤 샘플링 (중복 허용)
# 나무1용 데이터: 800경기 (일부 중복)
# 나무2용 데이터: 800경기 (다른 조합)
# 나무3용 데이터: 800경기 (또 다른 조합)
# ...
# 나무200용 데이터: 800경기

# 2단계 — 각 나무 독립 학습
# 각 나무는 피처 전체가 아닌 랜덤으로 뽑은 일부 피처만 사용해 학습합니다. 
# 피처가 10개면 보통 √10 ≈ 3개만 사용합니다.

# 나무1: rank_diff, points_avg_lastn_diff, goals_avg_diff 만 사용 → 홈승
# 나무2: rank_home, goals_suf_avg_diff, rank_avg_diff 만 사용   → 무승부
# 나무3: points_ratio, rank_diff, goals_avg_lastn_diff 만 사용  → 홈승
# ...

# 3단계 — 다수결 투표
# 200개 나무의 결과를 모아 가장 많이 나온 클래스를 최종 답으로 선택합니다.

# 분류 모델이기때문에 승/패/무 예측만 가능하다

In [ ]:
# poisson모델은 횟수를 예측하는 회귀 모델입니다
# 예를들어 home = 1.3(홈팀 예상 평균 득점)
# away = 0.8(어웨이팀 예상 평균 득점) 일때
# 피처 입력 → PoissonRegressor → λ_home = 1.3 (홈팀 예상 평균 득점)
#                             → λ_away = 0.8 (어웨이팀 예상 평균 득점)

# ↓

# 포아송 분포로 모든 경우의 수 확률 계산:
#   홈 0골 × 어웨이 0골 → 무승부 확률
#   홈 1골 × 어웨이 0골 → 홈 승 확률
#   홈 0골 × 어웨이 1골 → 홈 패 확률
#   ... (최대 10골까지)

# ↓

# 세 확률 중 가장 높은 것 → 최종 예측


# 모든 경우의 수를 확률 계산해서 가장 높은것을 최종 채택합니다
# 그렇기에 0-0, 0-1,2-1 처럼 스코어라인 확률도 계산이 가능한겁니다.